# Final Notebook

## Analyst Decisions

- Creating a Pivot Chart for each Brench containing the number of events for each catagory:
    1. Children: 0 to 12
    2. Teens : 13 to 17
    3. Adults : 18+

***Note: Might need to reclassify the Adults into three categories: Young Adults, Adults, and Seniors.***

- Then, making a conclusion page that will include:
   - Top 15 Branches
   - Lowest 15 branches
   - Librarys that have more space/ computer hubs (Resources)
   - Regestiration to the library: how many children/Adults are registered



In [35]:
# Import Lib
# %matplotlib notebook
import pandas as pd
import sweetviz as sv
import matplotlib.pyplot as plt #this is for Graph creation.
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import interact
import mplcursors


# Explore DF function
def explore (Data):
    print(Data.head(5))
    print(Data.info())
    M_Values = Data.isnull().sum().sort_values(ascending=False) 
    print(M_Values)
    print(Data.duplicated().sum())
    missing = M_Values.index[0]
    print(Data[Data[missing].isnull()])

# DataFrame creation

tpl_BGI_2023 = pd.read_csv("data/tpl-branch-general-information-2023.csv",index_col=0)
tpl_BSR_2024 = pd.read_csv("data/tpl-branch-space-rentals-2024.csv",index_col=0)
tpl_RABB = pd.read_csv("data/tpl-card-registrations-annual-by-branch.csv",index_col=0)
tpl_EFS2 = pd.read_csv("data/tpl-events-feed_source2.csv",index_col=0)
tpl_VABB = pd.read_csv("data/tpl-visits-annual-by-branch.csv",index_col=0)
tpl_WUABB_2018_2023 = pd.read_csv("data/tpl-workstation-usage-annual-by-branch-2018-2023.csv",index_col=0)
tpl_LCBCT = pd.read_csv("data/library-circulation-by-cardholder-type.csv",index_col=0)

tpl_BGI_2023.head(5)

,BranchCode,PhysicalBranch,BranchName,Address,PostalCode,Website,Telephone,SquareFootage,PublicParking,KidsStop,...,Workstations,ServiceTier,Lat,Long,NBHDNo,NBHDName,TPLNIA,WardNo,WardName,PresentSiteYear
_id,,,,,,,,,,,,,,,,,,,,,
1,AB,1,Albion,"1515 Albion Road, Toronto, ON, M9V 1B2",M9V 1B2,https://www.tpl.ca/albion,416-394-5170,29000,59,1.0,...,38.0,DL,43.739826,-79.584096,2.0,Mount Olive-Silverstone-Jamestown,1.0,1.0,Etobicoke North,2017.0
2,ACD,1,Albert Campbell,"496 Birchmount Road, Toronto, ON, M1K 1N8",M1K 1N8,https://www.tpl.ca/albertcampbell,416-396-8890,28957,45,0.0,...,36.0,DL,43.708019,-79.269252,120.0,Clairlea-Birchmount,1.0,20.0,Scarborough Southwest,1971.0
3,AD,1,Alderwood,"2 Orianna Drive, Toronto, ON, M8W 4Y1",M8W 4Y1,https://www.tpl.ca/alderwood,416-394-5310,7341,shared,0.0,...,7.0,NL,43.601944,-79.547252,20.0,Alderwood,0.0,3.0,Etobicoke-Lakeshore,1999.0
4,AG,1,Agincourt,"155 Bonis Avenue, Toronto, ON, M1T 3W6",M1T 3W6,https://www.tpl.ca/agincourt,416-396-8943,27000,86,0.0,...,42.0,DL,43.785167,-79.293430,118.0,Tam O'Shanter-Sullivan,0.0,22.0,Scarborough-Agincourt,1991.0
5,AH,1,Armour Heights,"2140 Avenue Road, Toronto, ON, M5M 4M7",M5M 4M7,https://www.tpl.ca/armourheights,416-395-5430,2988,shared,0.0,...,5.0,NL,43.739337,-79.421889,39.0,Bedford Park-Nortown,0.0,8.0,Eglinton-Lawrence,1982.0


## Cleaning Decisions:

#### For table ***tpl-branch-general-information-2023*** 
1. Clumns removed: 
    - Location:  
    'Address',
    'PostalCode',
    'Lat',
    'Long',
    'WardName',
    'NBHDNo',
    'WardNo',
    - Program Indicators: 
    'AdultLiteracyProgram',
    'LeadingReading',
    - Operational: 
    'SquareFootage',
    'PublicParking',
    'Website',
    'Telephone',
    'ServiceTier',
    'PresentSiteYear'

2. Raws that contain 'KidsStop' were kept, (might need to change)
3. Brench name: 'Jane/Dundas', was changed to its current name: 'Daniel G. Hill'
4. The new DF called: tpl_BGI_2023_cl 



In [2]:
def clean_branches(df):
    df =  df[df['KidsStop'].notnull()]
    drop_columns = [
    # Location
    'Address',
    'PostalCode',
    'Lat',
    'Long',
    'WardName',
    'NBHDNo',
    'WardNo',
    # Program Indicators
    'AdultLiteracyProgram',
    'LeadingReading',
    # Operational
    'SquareFootage',
    'PublicParking',
    'Website',
    'Telephone',
    'ServiceTier',
    'PresentSiteYear']
    df = df.drop(columns = drop_columns)
    return df

tpl_BGI_2023_cl = clean_branches(tpl_BGI_2023)

tpl_BGI_2023_cl['BranchName'] = tpl_BGI_2023_cl['BranchName'].str.replace(
    'Jane/Dundas', 'Daniel G. Hill'
)


tpl_BGI_2023_cl['BranchName'] = tpl_BGI_2023_cl['BranchName'].str.replace(
    'Perth/Dupont', 'Junction Triangle'
)

# branch_names = set(tpl_BGI_2023_cl['BranchName'].unique())
branch_names = set(tpl_BGI_2023_cl['BranchName'].astype(str).str.strip().str.lower())
# len(tpl_BGI_2023_cl['BranchName'].unique())
print(branch_names)
print(len(branch_names))

{'elmbrook park', 'riverdale', 'st. james town', 'fairview', 'danforth/coxwell', 'scarborough civic centre', 'st. lawrence', 'albert campbell', 'fort york', 'cliffcrest', 'wychwood', 'armour heights', 'barbara frum', 'morningside', 'taylor memorial', 'weston', 's. walter stewart', 'humber bay', 'locke', 'oakwood village library and arts centre', 'runnymede', 'brentwood', 'humberwood', 'malvern', 'evelyn gregory', 'thorncliffe', 'york woods', 'high park', 'cedarbrae', 'rexdale', 'woodview park', 'palmerston', 'beaches', 'kennedy/eglinton', 'eglinton square', 'forest hill', 'dufferin/st. clair', 'agincourt', 'guildwood', 'bloor/gladstone', 'victoria village', 'highland creek', 'city hall', 'burrows hall', 'black creek', 'richview', "ethennonnhawahstihnen'", 'maria a. shchuka', 'swansea memorial', 'goldhawk park', 'new toronto', 'northern elms', 'jones', 'deer park', 'alderwood', 'sanderson', 'todmorden room', 'long branch', 'dawes road', 'yorkville', 'jane/sheppard', 'northern district',

#### Investigation Notes 
- Perth/Dupont, was closed in 2025 and moved to a new location called 'Junction Triangle'

- Now Junction Triangle is at 305 Campbell

- Jane/ Dundas Name was changed to Daniel G. Hill

- Libraries that don't show up on the events table:
    - todmorden room, humber bay, dawes road, pleasant view (Temp Closed),flemingdon park (Temp Closed), centennial (Closed for 3 years), 


#### For table ***tpl-events-feed_source2*** 

1. Columns removed:
        'EventID',
        'StartTime',
        'EndTime',
        'StartDateLocal',
        'RegistrationClosed',
        'FeaturedImageUrl'

2. Have 96 Locations, when Nan is Online events

In [3]:
# Table tpl-events-feed_source2 clean 

# print(tpl_EFS2['LastUpdatedOn'].unique())

def clean_branches(df):
    # remove all non physical branches
    # df = df[df[''].notnull()]
    drop_columns = [
        'EventID',
        'StartTime',
        'EndTime',
        'StartDateLocal',
        'RegistrationClosed',
        'FeaturedImageUrl'
    ]
    df = df.drop(columns = drop_columns)
    # df = df[~df['LocationName'].str.contains('Junction', case=True)]
    return df

tpl_EFS2_cl = clean_branches(tpl_EFS2)
# print(tpl_EFS2_cl.head(5))
# print(tpl_EFS2_cl.info())


tpl_EFS2_cl['LocationName'] = tpl_EFS2_cl['LocationName'].str.replace("Dufferin/St.Clair", "Dufferin/St. Clair")

tpl_EFS2_cl[tpl_EFS2_cl['LocationName'].isnull()]
events_locations = set(tpl_EFS2_cl['LocationName'].astype(str).str.strip().str.lower())
events_locations
branch_names = {x.lower() for x in branch_names}
diff_a = branch_names - events_locations
print(diff_a)

{'flemingdon park', 'pleasant view', 'todmorden room', 'humber bay', 'centennial', 'dawes road'}


## Merging the events with the breanch indo table


In [4]:
merged_data = tpl_BGI_2023_cl.merge(
    tpl_EFS2_cl,
    left_on='BranchName',      # column name in left table
    right_on='LocationName',     # column name in right table
    how='right'           # join type
)

merged_data.info()
merged_data.head(5)
# merged_data.Audiences.unique() 

<class 'pandas.DataFrame'>
RangeIndex: 8512 entries, 0 to 8511
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   BranchCode          8443 non-null   str    
 1   PhysicalBranch      8443 non-null   float64
 2   BranchName          8443 non-null   str    
 3   KidsStop            8443 non-null   float64
 4   CLC                 8443 non-null   float64
 5   DIH                 8443 non-null   float64
 6   TeenCouncil         8443 non-null   float64
 7   YouthHub            8443 non-null   float64
 8   Workstations        8443 non-null   float64
 9   NBHDName            8443 non-null   str    
 10  TPLNIA              8443 non-null   float64
 11  Title               8512 non-null   str    
 12  LocationName        8449 non-null   str    
 13  Audiences           8512 non-null   str    
 14  Languages           8512 non-null   str    
 15  EventTypes          8512 non-null   str    
 16  IsRecurring      

,BranchCode,PhysicalBranch,BranchName,KidsStop,CLC,DIH,TeenCouncil,YouthHub,Workstations,NBHDName,...,Title,LocationName,Audiences,Languages,EventTypes,IsRecurring,IsFull,Status,RegistrationIsFull,LastUpdatedOn
0,CL,1.0,North York Central Library,1.0,1.0,1.0,1.0,1.0,98.0,Willowdale West,...,Tea & Entertainment,North York Central Library,"Adults (18+), Older Adults",English,"Culture, Arts & Entertainment",False,False,ACTIVE,False,2026-02-26T12:22:14
1,CL,1.0,North York Central Library,1.0,1.0,1.0,1.0,1.0,98.0,Willowdale West,...,Tea & Entertainment,North York Central Library,"Adults (18+), Older Adults",English,"Culture, Arts & Entertainment",False,False,ACTIVE,False,2026-02-26T12:22:14
2,SWS,1.0,S. Walter Stewart,1.0,1.0,0.0,0.0,1.0,39.0,Danforth Village - East York,...,D&D for Teens: On-going Campaign,S. Walter Stewart,Teens (13-17),English,"Crafts & Hobbies, Games & Sports",True,False,ACTIVE,False,2026-02-26T12:22:14
3,SWS,1.0,S. Walter Stewart,1.0,1.0,0.0,0.0,1.0,39.0,Danforth Village - East York,...,S. Walter Stewart Seniors Social,S. Walter Stewart,Older Adults,English,"Crafts & Hobbies, Games & Sports",True,False,ACTIVE,False,2026-02-26T12:22:14
4,PL,1.0,Parliament Street,0.0,0.0,0.0,0.0,1.0,12.0,Moss Park,...,Finding Recovery Through Exercise Skills and H...,Parliament Street,Adults (18+),English,Personal Development,True,False,ACTIVE,False,2026-02-26T12:22:14


- The merged table include the events and the branch information columes that were not droped.

- The next step it to create a Graph and clasifiy the events Audiances catagories as followes:
    - IsChildren : Matches both 'Preschool Children' and 'School Age Children'
    - IsTeens: Matches 'Teens (13-17)'
    - IsAdults: Matches 'Adults (18+)' explicitly, avoiding 'Older Adults' and 'Younger Adults'
    - IsSeniors: Seniors / Older Adults (Matches 'Older Adults')
    - IsYoungerAdults: Matches 'Younger Adults (18-24)'

In [5]:
merged_data.Audiences.unique() 

<StringArray>
[                                                                                             'Adults (18+), Older Adults',
                                                                                                           'Teens (13-17)',
                                                                                                            'Older Adults',
                                                                                                            'Adults (18+)',
                                                                                                'Preschool Children (0-5)',
                                                                                              'School Age Children (6-12)',
                                                                      'Adults (18+), Older Adults, Younger Adults (18-24)',
                                                                    'Preschool Children (0-5), School Age Children (6-

In [6]:
# Step 1: Creating columns

# 1. Children (Matches both 'Preschool Children' and 'School Age Children')
merged_data['IsChildren'] = merged_data['Audiences'].str.contains('Children', na=False)

# 2. Teens (Matches 'Teens (13-17)')
merged_data['IsTeens'] = merged_data['Audiences'].str.contains('Teens', na=False)

# 3. Seniors / Older Adults (Matches 'Older Adults')
merged_data['IsSeniors'] = merged_data['Audiences'].str.contains('Older Adults', na=False)

# 4. Adults (Matches 'Adults (18+)' explicitly, avoiding 'Older Adults' and 'Younger Adults')
merged_data['IsAdults'] = merged_data['Audiences'].str.contains(r'Adults \(18\+\)', na=False)

# OPTIONAL: If you want to track 'Younger Adults (18-24)' separately
merged_data['IsYoungerAdults'] = merged_data['Audiences'].str.contains('Younger Adults', na=False)

# Step 2: Group by brench and sum:

events_by_group = merged_data.groupby('BranchName')[
    ['IsChildren', 'IsTeens', 'IsAdults', 'IsYoungerAdults', 'IsSeniors']
].sum()



- There are still the Nan location events, those are the online events.
I wii add this to the merged table and create the final version of the events catagorized in each branch to the rlvent group age.

In [7]:
# Online events - all NaN location rows from events table
online_events = tpl_EFS2_cl[tpl_EFS2_cl['LocationName'].isnull()].copy()
online_events['LocationName'] = 'Online'

online_events['IsChildren'] = online_events['Audiences'].str.contains('Children', na=False)
online_events['IsTeens'] = online_events['Audiences'].str.contains('Teens', na=False)
online_events['IsSeniors'] = online_events['Audiences'].str.contains('Older Adults', na=False)
online_events['IsAdults'] = online_events['Audiences'].str.contains(r'Adults \(18\+\)', na=False)
online_events['IsYoungerAdults'] = online_events['Audiences'].str.contains('Younger Adults', na=False)

events_by_group_online = online_events.groupby('LocationName')[
    ['IsChildren', 'IsTeens', 'IsAdults', 'IsYoungerAdults','IsSeniors']
].sum()

events_by_group_online = events_by_group_online.rename_axis('BranchName')

events_by_group
events_by_group_online

# EBB - Events By Brench
EBB_final = pd.concat([events_by_group, events_by_group_online])

EBB_final

,IsChildren,IsTeens,IsAdults,IsYoungerAdults,IsSeniors
BranchName,,,,,
Agincourt,40,21,39,3,25
Albert Campbell,53,11,59,4,21
Albion,60,26,19,1,0
Alderwood,20,0,2,0,1
Amesbury Park,67,3,32,2,20
...,...,...,...,...,...
Woodview Park,118,4,35,0,65
Wychwood,34,1,9,1,8
York Woods,69,40,76,4,62


In [8]:
EBB_final.index.unique()

Index(['Agincourt', 'Albert Campbell', 'Albion', 'Alderwood', 'Amesbury Park',
       'Annette Street', 'Armour Heights', 'Barbara Frum', 'Beaches',
       'Bendale', 'Black Creek', 'Bloor/Gladstone', 'Brentwood', 'Bridlewood',
       'Brookbanks', 'Burrows Hall', 'Cedarbrae', 'City Hall', 'Cliffcrest',
       'College/Shaw', 'Danforth/Coxwell', 'Daniel G. Hill', 'Davenport',
       'Deer Park', 'Don Mills', 'Downsview', 'Dufferin/St. Clair',
       'Eatonville', 'Eglinton Square', 'Elmbrook Park',
       'Ethennonnhawahstihnen'', 'Evelyn Gregory', 'Fairview', 'Forest Hill',
       'Fort York', 'Gerrard/Ashdale', 'Goldhawk Park', 'Guildwood',
       'High Park', 'Highland Creek', 'Hillcrest', 'Humber Summit',
       'Humberwood', 'Jane/Sheppard', 'Jones', 'Junction Triangle',
       'Kennedy/Eglinton', 'Leaside', 'Lillian H. Smith', 'Locke',
       'Long Branch', 'Main Street', 'Malvern', 'Maria A. Shchuka', 'Maryvale',
       'McGregor Park', 'Mimico Centennial', 'Morningside', 'Mount

<!-- Graph Creation from here -->
## Graph Creations

1. Graph for each brench and its events catagorizes by group age

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import mplcursors


branch_selector = widgets.Dropdown(
    options=EBB_final.index.tolist(),
    description='Branch:'
)


def plot_branch(branch):
    data = EBB_final.loc[branch]
    data.plot(kind='bar', color=['#185FA5', '#854F0B', "#0F6E56", "#534AB7", "#993C1D"])
    plt.title(f'Events by age group — {branch}')
    plt.ylabel('Number of events')
    plt.xticks(rotation=0)
    plt.show()

# def plot_branch(branch):
#     plt.figure(figsize=(8, 5))

#     data = EBB_final.loc[branch]
#     ax = data.plot(        # ← change 1: save as ax
#         kind='bar',
#         color=['#185FA5', '#854F0B', "#0F6E56", "#534AB7", "#993C1D"],
#         # ax=plt.gca()
#     )
#     # data.plot(kind='bar', color=['#185FA5', '#854F0B', "#0F6E56", "#534AB7", "#993C1D"])
#     plt.title(f'Events by age group — {branch}')
#     plt.ylabel('Number of events')
#     plt.xticks(rotation=0)

#     mplcursors.cursor(ax, hover=True) 
#     plt.show()

widgets.interact(plot_branch, branch=branch_selector)

interactive(children=(Dropdown(description='Branch:', options=('Agincourt', 'Albert Campbell', 'Albion', 'Alde…

<function __main__.plot_branch(branch)>

2. Graph for each brench on the events thy offer that concerns 
 public speaking workshops, senior digital 
literacy, volunteer fairs, or youth leadership programming

In [20]:
unique_categories = merged_data['EventTypes'].str.split(',').explode().str.strip().unique()
exp = merged_data.groupby(['BranchName', 'EventTypes'])
# Sort them alphabetically to make it easier to read

print(unique_categories)
exp.head(5)

<StringArray>
[                          'Culture',              'Arts & Entertainment',
                  'Crafts & Hobbies',                    'Games & Sports',
              'Personal Development',     'Reading Programs & Storytimes',
             'Science & Engineering',                 'Health & Wellness',
       'Book Clubs & Writers Groups',      'Civic & Community Engagement',
                 'Coding & Robotics',              'Conversation Circles',
                  'Personal Finance', 'Small Business & Entrepreneurship',
               'Career & Job Search',           'Author Talks & Lectures',
 'Computer Basics & Office Software',                       '3D Printing',
                    'Audio & Visual',                    'Digital Safety',
            'Library Visits & Tours',          'Nature & the Environment',
           'Artificial Intelligence',               'History & Genealogy',
                 'Language Learning',                          'Exhibits',
           

,BranchCode,PhysicalBranch,BranchName,KidsStop,CLC,DIH,TeenCouncil,YouthHub,Workstations,NBHDName,...,IsRecurring,IsFull,Status,RegistrationIsFull,LastUpdatedOn,IsChildren,IsTeens,IsSeniors,IsAdults,IsYoungerAdults
0,CL,1.0,North York Central Library,1.0,1.0,1.0,1.0,1.0,98.0,Willowdale West,...,False,False,ACTIVE,False,2026-02-26T12:22:14,False,False,True,True,False
1,CL,1.0,North York Central Library,1.0,1.0,1.0,1.0,1.0,98.0,Willowdale West,...,False,False,ACTIVE,False,2026-02-26T12:22:14,False,False,True,True,False
2,SWS,1.0,S. Walter Stewart,1.0,1.0,0.0,0.0,1.0,39.0,Danforth Village - East York,...,True,False,ACTIVE,False,2026-02-26T12:22:14,False,True,False,False,False
3,SWS,1.0,S. Walter Stewart,1.0,1.0,0.0,0.0,1.0,39.0,Danforth Village - East York,...,True,False,ACTIVE,False,2026-02-26T12:22:14,False,False,True,False,False
4,PL,1.0,Parliament Street,0.0,0.0,0.0,0.0,1.0,12.0,Moss Park,...,True,False,ACTIVE,False,2026-02-26T12:22:14,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8498,BF,1.0,Barbara Frum,0.0,1.0,0.0,1.0,1.0,30.0,Englemount-Lawrence,...,True,False,ACTIVE,NaN,2026-02-26T12:22:14,False,True,False,False,False
8504,FV,1.0,Fairview,1.0,1.0,0.0,1.0,1.0,55.0,Don Valley Village,...,False,False,ACTIVE,False,2026-02-26T12:22:14,True,True,True,True,True
8507,TH,1.0,Thorncliffe,1.0,0.0,0.0,0.0,1.0,19.0,Thorncliffe Park,...,False,False,ACTIVE,False,2026-02-26T12:22:14,False,False,True,True,True
8509,AB,1.0,Albion,1.0,1.0,1.0,1.0,1.0,38.0,Mount Olive-Silverstone-Jamestown,...,False,False,ACTIVE,False,2026-02-26T12:22:14,False,False,False,True,False


In [26]:
Events_by_type = merged_data[[
    'EventTypes', 
    'BranchName', 
    'IsChildren', 
    'IsTeens', 
    'IsAdults', 
    'IsYoungerAdults', 
    'IsSeniors'
]].copy()

Graph2 = Events_by_type.groupby(['BranchName', 'EventTypes']).sum()
Graph2

IsChildren  \
BranchName EventTypes                                                       
Agincourt  3D Printing                                                  0   
           Artificial Intelligence                                      0   
           Audio & Visual                                               1   
           Audio & Visual, Coding & Robotics, Computer Bas...           1   
           Audio & Visual, Computer Basics & Office Software            0   
...                                                                   ...   
Yorkville  Exhibits                                                     0   
           Health & Wellness                                            0   
           Nature & the Environment                                     1   
           Reading Programs & Storytimes                                8   
           Science & Engineering                                        1   

                                                               IsTeens  \
BranchName EventTypes                                                    
Agincourt  3D Printing                                               2   
           Artificial Intelligence                                   0   
           Audio & Visual                                            0   
           Audio & Visual, Coding & Robotics, Computer Bas...        1   
           Audio & Visual, Computer Basics & Office Software         0   
...                                                                ...   
Yorkville  Exhibits                                                  0   
           Health & Wellness                                         0   
           Nature & the Environment                                  0   
           Reading Programs & Storytimes                             0   
           Science & Engineering                                     0   

                                                               IsAdults  \
BranchName EventTypes                                                     
Agincourt  3D Printing                                                2   
           Artificial Intelligence                                    6   
           Audio & Visual                                             1   
           Audio & Visual, Coding & Robotics, Computer Bas...         1   
           Audio & Visual, Computer Basics & Office Software          4   
...                                                                 ...   
Yorkville  Exhibits                                                   2   
           Health & Wellness                                          1   
           Nature & the Environment                                   0   
           Reading Programs & Storytimes                              0   
           Science & Engineering                                      0   

                                                               IsYoungerAdults  \
BranchName EventTypes                                                            
Agincourt  3D Printing                                                       2   
           Artificial Intelligence                                           0   
           Audio & Visual                                                    0   
           Audio & Visual, Coding & Robotics, Computer Bas...                0   
           Audio & Visual, Computer Basics & Office Software                 0   
...                                                                        ...   
Yorkville  Exhibits                                                          0   
           Health & Wellness                                                 0   
           Nature & the Environment                                          0   
           Reading Programs & Storytimes                                     0   
           Science & Engineering                                             0   

                                                       

In [ ]:
# Data Perparation
processed_df = Graph2.reset_index()
processed_df['EventTypes'] = processed_df['EventTypes'].str.split(',')
processed_df = processed_df.explode('EventTypes')
processed_df['EventTypes'] = processed_df['EventTypes'].str.strip()

# Group it so we have clean sums per Branch/Type
final_counts = processed_df.groupby(['BranchName', 'EventTypes']).sum()

# 2. Create the Dropdown using ONLY unique Branch names
branch_list = sorted(processed_df['BranchName'].unique())
branch_selector = widgets.Dropdown(
    options=branch_list,
    value=branch_list[0], # Default value
    description='Branch:',
    disabled=False,
)


# 3. The Plotting Function
def plot_branch(branch):
    # Select just the data for the chosen branch
    branch_data = final_counts.xs(branch, level='BranchName')
    
    # Create the plot
    ax = branch_data.plot(kind='barh', stacked=True, figsize=(12, 7), colormap='viridis')
    
    plt.title(f'Event Types Distribution: {branch} Branch', fontsize=14, pad=20)
    plt.xlabel('Number of Programs')
    plt.ylabel('Event Type')
    plt.legend(title='Demographics', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

interact(plot_branch, branch=branch_selector);

interactive(children=(Dropdown(description='Branch:', options=('Agincourt', 'Albert Campbell', 'Albion', 'Alde…

In [ ]:

# 1. Prepare data (Reset index so 'EventTypes' and 'BranchName' are columns)
# We melt the demographic columns so Plotly can 'stack' them and show labels
final_df = final_counts.reset_index()
melted_df = final_df.melt(
    id_vars=['BranchName', 'EventTypes'], 
    value_vars=['IsChildren', 'IsTeens', 'IsAdults', 'IsYoungerAdults', 'IsSeniors'],
    var_name='Demographic', 
    value_name='ProgramCount'
)

# 2. The Interactive Plotting Function
def plot_branch_interactive(branch):
    # Filter for the selected branch
    branch_data = melted_df[melted_df['BranchName'] == branch]
    
    # Create Plotly Express horizontal bar chart
    fig = px.bar(
        branch_data, 
        y='EventTypes', 
        x='ProgramCount', 
        color='Demographic',
        orientation='h',
        title=f'Event Types Distribution: {branch} Branch',
        labels={'ProgramCount': 'Number of Programs', 'EventTypes': 'Event Type'},
        height=700,
        color_discrete_sequence=px.colors.qualitative.Plotly # Good distinct colors
    )
    
    # Improve the layout and hover behavior
    fig.update_layout(
        hovermode="y unified", # Shows all demographics for that bar when hovering
        xaxis_title="Count",
        yaxis={'categoryorder':'total ascending'}, # Sorts bars by size
        legend_title="Target Audience"
    )
    
    fig.show()

# 3. Create the widget (using the same list from before)
branch_list = sorted(melted_df['BranchName'].unique())
interact(plot_branch_interactive, branch=widgets.Dropdown(options=branch_list, description='Branch:'));

interactive(children=(Dropdown(description='Branch:', options=('Agincourt', 'Albert Campbell', 'Albion', 'Alde…

In [12]:
tpl_RABB.info()
tpl_VABB.info()

<class 'pandas.DataFrame'>
RangeIndex: 1275 entries, 1 to 1275
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Year           1275 non-null   int64
 1   BranchCode     1275 non-null   str  
 2   Registrations  1275 non-null   int64
dtypes: int64(2), str(1)
memory usage: 30.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 1233 entries, 1 to 1233
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Year        1233 non-null   int64
 1   BranchCode  1233 non-null   str  
 2   Visits      1233 non-null   int64
dtypes: int64(2), str(1)
memory usage: 29.0 KB
